In [1]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output

from resources import find_tilt_error
from resources.cut_off_edges import cut_off_edges
from resources.find_rotation_error import find_rotation_error_by_profile
from resources.find_tilt_error import find_tilt_error_by_profile
from resources.rotate_image import rotate_volume
from resources.standarize_image import standardize_image

project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

In [2]:
data = np.load(
    project_dir / "data" / "processed_profiles.npz",
    allow_pickle=False,
)
print(data.files)
uids = data["uid"]
y = data["is_pathologic"]

x_profile=data['x_profile']
y_profile=data['y_profile']
z_profile=data['z_profile']
rotation_x_profile=data['rotation_x_profile']
tilt_x_profiles=data['tilt_x_profiles']
pitch_y_profiles=data['pitch_y_profiles']


['uid', 'is_pathologic', 'x_profile', 'y_profile', 'z_profile', 'rotation_x_profile', 'tilt_x_profiles', 'pitch_y_profiles']


In [3]:
X = np.concatenate(
    [
        x_profile,
        y_profile,
        z_profile,
        rotation_x_profile,
    ],
    axis=1,
)

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", np.unique(y_train, return_counts=True))
print("y_test:", np.unique(y_test, return_counts=True))

X_train: (1089, 480)
X_test: (273, 480)
y_train: (array([0, 1], dtype=int8), array([492, 597]))
y_test: (array([0, 1], dtype=int8), array([123, 150]))


In [5]:
import copy
import numpy as np
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss, accuracy_score

X_train_tensor = torch.tensor(
    X_train,
    dtype=torch.float32,
)

X_test_tensor = torch.tensor(
    X_test,
    dtype=torch.float32,
)

y_train_tensor = torch.tensor(
    y_train,
    dtype=torch.float32,
)

y_test_tensor = torch.tensor(
    y_test,
    dtype=torch.float32,
)
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor,
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
)

In [6]:
class ProfileMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(input_size, 1560),
            nn.LayerNorm(1560),
            nn.ReLU(),
            nn.Dropout(0.85),

            nn.Linear(1560, 128),
            nn.LayerNorm(128),
            nn.ReLU(),
            nn.Dropout(0.8),

            nn.Linear(128, 32),
            nn.LayerNorm(32),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(32, 1),
        )

    def forward(self, x):
        return self.network(x).squeeze(1)

In [7]:
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = ProfileMLP(
    input_size=X_train.shape[1],
).to(device)

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=0.0001,
)

print(model)
print("Urządzenie:", device)

ProfileMLP(
  (network): Sequential(
    (0): Linear(in_features=480, out_features=1560, bias=True)
    (1): LayerNorm((1560,), eps=1e-05, elementwise_affine=True)
    (2): ReLU()
    (3): Dropout(p=0.85, inplace=False)
    (4): Linear(in_features=1560, out_features=128, bias=True)
    (5): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
    (6): ReLU()
    (7): Dropout(p=0.8, inplace=False)
    (8): Linear(in_features=128, out_features=32, bias=True)
    (9): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
    (10): ReLU()
    (11): Dropout(p=0.4, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)
Urządzenie: cuda


In [8]:
epochs = 40000
patience = 500

best_validation_loss = float("inf")
best_model_state = copy.deepcopy(model.state_dict())

epochs_without_improvement = 0

for epoch in range(epochs):
    # Trening
    model.train()

    train_loss_sum = 0.0
    train_samples = 0

    for X_batch, y_batch in train_loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        logits = model(X_batch)
        loss = criterion(logits, y_batch)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * len(X_batch)
        train_samples += len(X_batch)

    train_loss = train_loss_sum / train_samples

    # Walidacja
    model.eval()

    validation_loss_sum = 0.0
    validation_samples = 0

    with torch.no_grad():
        for X_batch, y_batch in test_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            logits = model(X_batch)
            loss = criterion(logits, y_batch)

            validation_loss_sum += loss.item() * len(X_batch)
            validation_samples += len(X_batch)

    validation_loss = (
        validation_loss_sum / validation_samples
    )

    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        best_model_state = copy.deepcopy(
            model.state_dict()
        )
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(
        f"Epoka {epoch + 1:3d}/{epochs} | "
        f"train: {train_loss:.4f} | "
        f"validation: {validation_loss:.4f} | "
        f"best: {best_validation_loss:.4f}"
    )

    if epochs_without_improvement >= patience:
        print("Early stopping")
        break

model.load_state_dict(best_model_state)
model.eval()

with torch.no_grad():
    logits = model(
        X_test_tensor.to(device)
    )

    probabilities = torch.sigmoid(logits)
    probabilities = probabilities.cpu().numpy()

predictions = (probabilities >= 0.5).astype(int)

print(
    "Log loss:",
    log_loss(y_test, probabilities),
)

print(
    "Accuracy:",
    accuracy_score(y_test, predictions),
)

Epoka   1/40000 | train: 0.7027 | validation: 0.6909 | best: 0.6909
Epoka   2/40000 | train: 0.7059 | validation: 0.6908 | best: 0.6908
Epoka   3/40000 | train: 0.7005 | validation: 0.6882 | best: 0.6882
Epoka   4/40000 | train: 0.6934 | validation: 0.6883 | best: 0.6882
Epoka   5/40000 | train: 0.6976 | validation: 0.6882 | best: 0.6882
Epoka   6/40000 | train: 0.6865 | validation: 0.6884 | best: 0.6882
Epoka   7/40000 | train: 0.6972 | validation: 0.6884 | best: 0.6882
Epoka   8/40000 | train: 0.6962 | validation: 0.6887 | best: 0.6882
Epoka   9/40000 | train: 0.6875 | validation: 0.6884 | best: 0.6882
Epoka  10/40000 | train: 0.6920 | validation: 0.6882 | best: 0.6882
Epoka  11/40000 | train: 0.6940 | validation: 0.6885 | best: 0.6882
Epoka  12/40000 | train: 0.6909 | validation: 0.6882 | best: 0.6882
Epoka  13/40000 | train: 0.6904 | validation: 0.6882 | best: 0.6882
Epoka  14/40000 | train: 0.6893 | validation: 0.6883 | best: 0.6882
Epoka  15/40000 | train: 0.6890 | validation: 0.

In [9]:
from pathlib import Path
from datetime import datetime
import torch


models_dir = project_dir / "models"
models_dir.mkdir(parents=True, exist_ok=True)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

model_path = models_dir / f"simple_profile_pitch_xyz_p_rot_x_cnn_{timestamp}_{best_validation_loss}.pth"

torch.save(
    model.state_dict(),
    model_path,
)

print(f"Zapisano model: {model_path} {best_validation_loss}")

Zapisano model: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\models\simple_profile_pitch_xyz_p_rot_x_cnn_2026-09-14_12-18-18_0.5271786499372769.pth 0.5271786499372769
